 ##### Step 1: First I am loading all the source tables

In [2]:
from pyspark.sql.functions import *

df_customers = spark.read.table("bronze_olist_customers_dataset")
df_order_items = spark.read.table("bronze_olist_order_items_dataset")
df_order_payments = spark.read.table("bronze_olist_order_payments_dataset")
df_orders = spark.read.table("bronze_olist_orders_dataset")
df_products = spark.read.table("bronze_olist_products_dataset")


StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 4, Finished, Available, Finished, False)

In [ ]:
# Checking if all tables are loaded in dataframe correctly
# display(df_customers)
# display(df_order_items)
# display(df_order_payments)
# display(df_orders)
# display(df_products)

##### Step 2: Handling customers without customer_id (Basically null customer_id)

In [3]:
df_customers_clean = df_customers.filter("customer_id is not null")

#Note: We can use any other way also if business users need customer 
#records without id also for that we can then predecide and keep some id for such users

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 5, Finished, Available, Finished, False)

##### Step 3: Casting columns as per requirement for order_items

In [4]:
from pyspark.sql.functions import col 
df_order_items_clean = df_order_items.withColumn('shipping_limit_date',col('shipping_limit_date').cast('date'))\
.withColumn('price',col('price').cast('float'))\
.withColumn('freight_value',col('freight_value').cast('float'))

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 6, Finished, Available, Finished, False)

##### Step 4: Handling negative and null payment values and casting as per need

In [5]:
from pyspark.sql.functions import col

df_order_payments_clean = df_order_payments.filter(
    (col("payment_value") >= 0) & (col("payment_value").isNotNull())
).withColumn('payment_installments',col('payment_installments').cast('int'))\
.withColumn('payment_value',col('payment_value').cast('float'))

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 7, Finished, Available, Finished, False)

##### Step 5: Changing date columns for orders as per required format

In [6]:
from pyspark.sql.functions import to_timestamp
df_orders_clean = df_orders\
.withColumn('order_purchase_timestamp',to_timestamp(col('order_purchase_timestamp'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_approved_at',to_timestamp(col('order_approved_at'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_delivered_carrier_date',to_timestamp(col('order_delivered_carrier_date'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_delivered_customer_date',to_timestamp(col('order_delivered_customer_date'),'yyyy-MM-dd HH:mm:ss'))\
.withColumn('order_estimated_delivery_date',to_timestamp(col('order_estimated_delivery_date'),'yyyy-MM-dd HH:mm:ss'))\
.dropDuplicates(["order_id"])

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 8, Finished, Available, Finished, False)

##### Step 6: For products casting of numeric columns is done

In [7]:
df_products_clean = df_products\
.withColumn('product_name_lenght',col('product_name_lenght').cast('int'))\
.withColumn('product_description_lenght',col('product_description_lenght').cast('int'))\
.withColumn('product_photos_qty',col('product_photos_qty').cast('int'))\
.withColumn('product_weight_g',col('product_weight_g').cast('float'))\
.withColumn('product_length_cm',col('product_length_cm').cast('float'))\
.withColumn('product_height_cm',col('product_height_cm').cast('float'))\
.withColumn('product_width_cm',col('product_width_cm').cast('float'))


StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 9, Finished, Available, Finished, False)

##### Step 7: Writing clean dataframes to Silver tables

In [8]:
df_customers_clean.write.mode("overwrite").saveAsTable("silver_customers")
df_order_items_clean.write.mode("overwrite").saveAsTable("silver_order_items")
df_order_payments_clean.write.mode("overwrite").saveAsTable("silver_order_payments")
df_orders_clean.write.mode("overwrite").saveAsTable("silver_orders")
df_products_clean.write.mode("overwrite").saveAsTable("silver_products")


StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 10, Finished, Available, Finished, False)

##### Loading Silver tables in dataframes

In [9]:
df_customer_silver = spark.read.table("silver_customers")
df_order_items_silver = spark.read.table("silver_order_items")
df_order_payments_silver = spark.read.table("silver_order_payments")
df_orders_silver = spark.read.table("silver_orders")
df_products_silver = spark.read.table("silver_products")


StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 11, Finished, Available, Finished, False)

##### Aggregating payment values for single order_id

In [10]:
from pyspark.sql.functions import col,sum,round
df_payments_agg = df_order_payments_silver.groupBy(col("order_id"))\
.agg(round(sum(col("payment_value")),2).alias("total_payment_value"))

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 12, Finished, Available, Finished, False)

##### Aggregating order items for single order_id

In [11]:
from pyspark.sql.functions import sum , count

df_items_agg = df_order_items_silver.groupBy("order_id") \
    .agg(
        sum("price").alias("total_item_price"),
        sum("freight_value").alias("total_freight"),
        count("order_item_id").alias("total_items")
    )

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 13, Finished, Available, Finished, False)

##### Joining datasets (customers + orders+ payments + order_items)

In [12]:
df_joined = df_orders_silver.alias('o').join\
(df_customer_silver.alias('c'),"customer_id",'inner')
#display(df_joined)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 14, Finished, Available, Finished, False)

In [13]:
df_joined = df_orders_silver.alias('o').join\
(df_customer_silver.alias('c'),"customer_id",'inner')
#display(df_joined)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 15, Finished, Available, Finished, False)

In [14]:
df_joined = df_joined.alias('j').join\
(df_payments_agg.alias('p'),"order_id",'left')
#display(df_joined)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 16, Finished, Available, Finished, False)

In [15]:
df_joined = df_joined.alias('j').join\
(df_items_agg.alias('oi'),"order_id",'left')
#display(df_joined)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 17, Finished, Available, Finished, False)

##### Adding Business Columns

In [16]:
df_joined = df_joined.withColumn(
    "order_value",round((col("total_item_price") + col("total_freight")),2)
)\
.withColumn(
    "is_delivered",(col("order_status")=='delivered').cast("int")
)
#display(df_joined)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 18, Finished, Available, Finished, False)

##### Handling nulls in value columns

In [17]:
df_joined =df_joined.fillna({
    "total_payment_value":0,
    "order_value":0
})

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 19, Finished, Available, Finished, False)

##### Selecting only required columns from the df_joined

In [24]:
from pyspark.sql.functions import current_timestamp

df_final = df_joined.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "total_payment_value",
    "total_items",
    "order_value",
    "is_delivered"
).withColumn(
    "insert_ts", current_timestamp()
).withColumn(
    "update_ts", current_timestamp()
)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 26, Finished, Available, Finished, False)

##### Finally writing to Silver table

In [27]:
df_final.write.mode("overwrite").saveAsTable("silver_orders_enriched")

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 29, Finished, Available, Finished, False)

##### Validation check

In [18]:
display(df_final.groupBy("order_id").count().filter("count > 1"))

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d8bd8fe5-c7be-4254-bb35-d828a03c64f6)

In [21]:
df_final.filter("order_id IS NULL").count()

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 23, Finished, Available, Finished, False)

0

In [23]:
df_final.filter("total_payment_value < 0").count()

StatementMeta(, cbc945ed-3f3c-4b7f-97d4-7818f25b7f89, 25, Finished, Available, Finished, False)

0

##### Creating Temp View

In [28]:
df_final.createOrReplaceTempView("temp_incremental")

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 30, Finished, Available, Finished, False)

##### Writing Merge

In [30]:
%%sql
MERGE INTO silver_orders_enriched AS target
USING temp_incremental AS source
ON target.order_id = source.order_id

WHEN MATCHED AND (
    target.customer_id <> source.customer_id OR
    target.order_status <> source.order_status OR
    target.order_purchase_timestamp <> source.order_purchase_timestamp OR
    target.total_payment_value <> source.total_payment_value OR
    target.total_items <> source.total_items OR
    target.order_value <> source.order_value OR
    target.is_delivered <> source.is_delivered
)
THEN UPDATE SET
    target.customer_id = source.customer_id,
    target.order_status = source.order_status,
    target.order_purchase_timestamp = source.order_purchase_timestamp,
    target.total_payment_value = source.total_payment_value,
    target.total_items = source.total_items,
    target.order_value = source.order_value,
    target.is_delivered = source.is_delivered,
    target.update_ts = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp,
    total_payment_value,
    total_items,
    order_value,
    is_delivered,
    insert_ts,
    update_ts
)
VALUES (
    source.order_id,
    source.customer_id,
    source.order_status,
    source.order_purchase_timestamp,
    source.total_payment_value,
    source.total_items,
    source.order_value,
    source.is_delivered,
    source.insert_ts,
    source.update_ts
)

StatementMeta(, bb036c60-57cb-4fa3-ac33-6a8e74880388, 32, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>